This testing serves two purposes:
1. User lowers the pen to touch the fiducial marker to know the pen tip location wrt stage, hence stage coordinates for pen tip after reinstallation.
2. Jogging down operation also gives us the relative z distance between the pen tip after calibration to the slide 

10/28/2025 update 
The above proposed pen stage calibraion is on hold due to vulnerability from aligning the fiducial mark (A and B) to pen tip with bare human eyes. Currently testing if we can spot something onto the chip and align the center of the camera with the center of that spot. Same offset logic holds. 
Operation process sequence:
1. calibrate arm 
2. calibrate stage 
3. calibrate first well 
4. calibration pen/camera offset 

In [1]:
import sys
sys.path.append("C:\\Users\\NikonTE300CE\\Desktop\\automated-sca\\src")

from stage import Stage
from plate import Plate
from chip import Chip
from stepper import *
from arm import *
import serial
import time


In [2]:
arm = Arm("COM5")
zm = arm.zmotor

In [3]:
# calibrate the arm origin
zm.calibrateOrigin()

In [4]:
c = Chip()
p = Plate()
# p = Plate(2,3,40000) # roughly the diameter for 6-well plate, a little off
ser = serial.Serial(port="COM4", baudrate=9600, timeout=0.1) 
s = Stage(p, c, ser)

In [5]:
# move stage to bottom right corner to calibrate origin
s.calibOrigin()

In [22]:
zm.moveToZInUM(-21000)

In [23]:
# user lower the pen into the well and up the arm to the safe z height, move stage until the pen is directly above the chip to where you want to spot

# user define parameters 
safe_z = -21000 # in um 
in_well_z = -30000 # in um 
spot_z = -41700 # in um 
spot_dwell_time = 2 # in seconds 

# # getting solution
# zm.moveToZInUM(safe_z)  # move to safe height
# zm.moveToZInUM(in_well_z)  # move to in well height
# time.sleep(1)
# zm.moveToZInUM(safe_z)

In [20]:
# get first well position 
b = s.getStageXY()
print("First well stage position:", b)

firstwellpos = tuple(b)
# save to tuple to do calculations later
print("First well position set to:", firstwellpos)

First well stage position: [-119710, -10023]
First well position set to: (-119710, -10023)


In [12]:
# user move the stage for spotting onto the chip
zm.moveToZInUM(spot_z + 2000) # buffer/safety check

In [19]:
# # if its good to spot
# zm.moveToZInUM(spot_z)
# time.sleep(spot_dwell_time)
# zm.moveToZInUM(safe_z) # back to safe z

# save current stage position as pen location
pen_spot_pos = s.getStageXY()
print(f"Pen position on stage:, {pen_spot_pos}")

Pen position on stage:, [-71965, -23753]


In [24]:
# automated spotting process
# getting solution
zm.moveToZInUM(safe_z)  # move to safe height
s.moveToPos(b[0], b[1])  # move to first well position
zm.moveToZInUM(safe_z)  # move to safe height
zm.moveToZInUM(in_well_z)  # move to in well height
time.sleep(1)
zm.moveToZInUM(safe_z)
time.sleep(1)

s.moveToPos(pen_spot_pos[0], pen_spot_pos[1])  # move to pen spot position
zm.moveToZInUM(spot_z)
time.sleep(spot_dwell_time)
zm.moveToZInUM(safe_z) # back to safe z


In [ ]:
# # user move the stage until the center mark in the camera frame is in the center of the fiducial marker
# fiducial_stage_coords = s.getStageXY()
# print(f"Fiducial marker stage coordinates: {fiducial_stage_coords}")

Fiducial marker stage coordinates: [-40410, -41050]


In [25]:
# user move the stage until the center mark in the camera frame is aligned with the center of the spot from the pen
cam_spot_pos = s.getStageXY()
print(f"Camera spot stage coordinates: {cam_spot_pos}")

Camera spot stage coordinates: [-83484, -10884]


In [ ]:
# # user lowers the pen to touch the fiducial marker to know pen stage location
# pen_stage_coords = s.getStageXY()
# print(f"Pen tip stage coordinates: {pen_stage_coords}")

Pen tip stage coordinates: [-29847, -44195]


In [26]:
pen_cam_offset_xy = [cam_spot_pos - pen_spot_pos for cam_spot_pos, pen_spot_pos in zip(cam_spot_pos, pen_spot_pos)]
print(f"Pen offset (stage coords) from fiducial marker: {pen_cam_offset_xy}")

Pen offset (stage coords) from fiducial marker: [-11519, 12869]


In [27]:
# Move stage to see first channel on camera, could find the first channel near either A or B on the chip
first_channel = s.getStageXY()
print("First channel stage position:", first_channel)

firstchannelpos = tuple(first_channel)
# secondchannelpos = (firstchannelpos[0], firstchannelpos[1]  + 260)
channel_positions = []

# Generate first 5 channels, each spaced by 260 µm in y direction
for i in range(5):
    channel_positions.append((firstchannelpos[0] - i * 260, firstchannelpos[1])) 

print("Channel positions:", channel_positions)

First channel stage position: [-80629, -10883]
Channel positions: [(-80629, -10883), (-80889, -10883), (-81149, -10883), (-81409, -10883), (-81669, -10883)]


In [17]:
zm.moveToZInUM(-20000)

In [ ]:
# # Move stage and lower the pen into the first well 
# b = s.getStageXY()
# print("First well stage position:", b)

# firstwellpos = tuple(b)
# # save to tuple to do calculations later
# print("First well position set to:", firstwellpos)


First well stage position: [-120634, -1538]
First well position set to: (-120634, -1538)


In [28]:
import time
from plate import Plate384
from stepper import *
from arm import *

def run_dispense_sequence(Stage, plate, channel_positions, firstwellpos, cam_spot_pos, 
                          pen_spot_pos, safe_z, in_well_z, spot_z):
    """
    Full calibration + dispensing sequence.
    
    Args:
        stage: Stage() instance
        plate: Plate() instance (e.g. Plate384())
        first_channel_pos: (x,y) tuple of first channel (user calibrated)
        firstwellpos: (x,y) tuple of first well (user calibrated)
        fiducial_stage_coords: (x,y) stage coords of fiducial marker
        pen_stage_coords: (x,y) stage coords of pen tip aligned to fiducial
        safe_z: Z height above surface (microns or steps) to move safely
        dip_depth: Z distance to dip below safe height into well/channel
    """

    # --- Calculate offsets ---
    # pen_offset_xy = [f - p for f, p in zip(fiducial_stage_coords, pen_stage_coords)]
    # print(f"[Calibration] Pen offset relative to fiducial: {pen_offset_xy}")

    origin_xy = firstwellpos
    diam = 4500  # spacing for 384-well plate

    def well_id_to_index(well_id):
        row = ord(well_id[0].upper()) - ord('A')
        col = int(well_id[1:]) - 1
        return row, col

    def get_well_xy(plate, well_id, origin_xy=(0, 0)):
        row, col = well_id_to_index(well_id)
        x = origin_xy[0] + col * diam
        y = origin_xy[1] + row * diam
        return (x, y)

    # Example run with A1 then channel
    well_id = "A1"
    well_xy = get_well_xy(plate, well_id, origin_xy)


    for i, channel_xy in enumerate(channel_positions):
        pen_cam_offset_xy = [cam_spot_pos - pen_spot_pos for cam_spot_pos, pen_spot_pos in zip(cam_spot_pos, pen_spot_pos)]
        channel_xy_pen = (channel_xy[0] - pen_cam_offset_xy[0], channel_xy[1] - pen_cam_offset_xy[1])

        # --- Well step ---
        print(f"[Dispense] Moving to well {well_id} at {well_xy}")
        zm.moveToZInUM(safe_z)  # move to safe height
        s.moveToPos(*well_xy)
        zm.moveToZInUM(in_well_z)  # dip down
        time.sleep(1)               # wait to get solution
        zm.moveToZInUM(safe_z)     # retract

        time.sleep(1)

        # --- Channel step ---
        print(f"[Dispense] Moving to channel at {channel_xy_pen}")
        s.moveToPos(*channel_xy_pen)
        # print("chip offset:", chip_offset)
        zm.moveToZInUM(spot_z) # move onto chip   
        time.sleep(1)               # wait to deposit solution
        zm.moveToZInUM(safe_z)    

        time.sleep(1)

    print("[Done] Dispense sequence complete.")


In [29]:
run_dispense_sequence(s, Plate384(), channel_positions, firstwellpos, cam_spot_pos, pen_spot_pos, safe_z, in_well_z, spot_z)

[Dispense] Moving to well A1 at (-119710, -10023)
[Dispense] Moving to channel at (-69110, -23752)
[Dispense] Moving to well A1 at (-119710, -10023)
[Dispense] Moving to channel at (-69370, -23752)
[Dispense] Moving to well A1 at (-119710, -10023)
[Dispense] Moving to channel at (-69630, -23752)
[Dispense] Moving to well A1 at (-119710, -10023)
[Dispense] Moving to channel at (-69890, -23752)
[Dispense] Moving to well A1 at (-119710, -10023)
[Dispense] Moving to channel at (-70150, -23752)
[Done] Dispense sequence complete.
